In [ ]:
# Cell 1: Setup & Module Imports
import os
import sys
import torch
import numpy as np

sys.path.append(os.path.abspath(".."))
import config
from src.models.rmlp_snn import RecurrentMLPSNN
from src.optimization.pruning import prune_synaptic_weights
from src.optimization.quantization import quantize_weights_to_6bit
from src.optimization.metrics import calculate_flops_vs_sops
from src.preprocessing.dataset import ECGSpikeDataset
from src.training.trainer import evaluate
from src.training.loss import MultiObjectiveSNNLoss

# Cell 2: Load Baseline Trained Model
device = config.DEVICE
model = RecurrentMLPSNN(input_dim=config.INPUT_DIM, beta=config.LIF_BETA).to(device)
weights_path = os.path.join(config.CHECKPOINT_DIR, "baseline_rmlp_snn.pth")

if os.path.exists(weights_path):
    model.load_state_dict(torch.load(weights_path, map_location=device))
    print("Successfully loaded baseline model weights.")
else:
    print("Baseline weights not found. Run Notebook 02 first to train the network.")

# Load Validation Set
val_dataset = ECGSpikeDataset(data_dir=config.DATA_RAW_DIR, delta_v=config.DELTA_V)
val_loader = torch.utils.data.DataLoader(val_dataset, batch_size=config.BATCH_SIZE, shuffle=False)
criterion = MultiObjectiveSNNLoss()

# Evaluate Dense Uncompressed Model
_, baseline_acc = evaluate(model, val_loader, criterion, device)
print(f"Uncompressed Float32 SNN Accuracy: {baseline_acc:.2f}%\n")

# Cell 3: Apply Synaptic Weight Pruning
print("--- Step 1: Synaptic Weight Pruning ---")
pruned_model = prune_synaptic_weights(model, threshold=config.PRUNING_THRESHOLD)
_, pruned_acc = evaluate(pruned_model, val_loader, criterion, device)
print(f"Pruned Model Accuracy: {pruned_acc:.2f}%\n")

# Cell 4: Apply 6-Bit Fixed-Point Quantization
print("--- Step 2: 6-Bit Weight Quantization ---")
quantized_model = quantize_weights_to_6bit(pruned_model)
_, quant_acc = evaluate(quantized_model, val_loader, criterion, device)
print(f"6-Bit Quantized SNN Accuracy: {quant_acc:.2f}%\n")

# Save Quantized Model Artifacts
os.makedirs(config.QUANTIZED_DIR, exist_ok=True)
torch.save(quantized_model.state_dict(), os.path.join(config.QUANTIZED_DIR, "quantized_6bit_rmlp_snn.pth"))

# Cell 5: Theoretical Energy Savings Calculation
print("--- Step 3: Theoretical Energy Benchmarking ---")
metrics = calculate_flops_vs_sops(
    quantized_model,
    avg_firing_rate=config.TARGET_FIRING_RATE,
    num_timesteps=config.TIME_STEPS,
    mac_energy_pj=config.MAC_ENERGY_PJ,
    ac_energy_pj=config.AC_ENERGY_PJ
)

# Display Summary Table
print("\n" + "="*45)
print(f" Baseline Accuracy (Float32):    {baseline_acc:.2f}%")
print(f" Compressed Accuracy (6-bit):   {quant_acc:.2f}%")
print(f" Active Synapses Remaining:      {metrics['active_synapses']:,}")
print(f" ANN Compute Energy per Beat:   {metrics['ann_energy_nj']:.2f} nJ")
print(f" SNN Compute Energy per Beat:   {metrics['snn_energy_nj']:.2f} nJ")
print(f" Total Compute Energy Savings:  {metrics['energy_savings_ratio']:.1f}x Reduction")
print("="*45)